# Module 2 Hands-On Tutorial: NumPy & Scientific Linear Algebra

**Applied Materials / Computational Materials — M.Tech/PhD Level**

### Tutorial goals

By the end of this notebook, you should be able to:

- Represent scientific quantities using NumPy arrays.
- Index, slice, reshape, and broadcast arrays.
- Replace unnecessary Python loops with vectorized NumPy operations.
- Perform matrix operations and solve linear systems.
- Compute determinants, rank, norms, and orthogonality.
- Understand and implement eigenvalue/eigenvector calculations.
- Use SVD for low-rank approximation and dimensional reduction.
- Understand QR decomposition as an optional advanced tool.
- Connect every computational operation to its underlying mathematics and a materials-science interpretation.

> **Philosophy:** Do not memorize NumPy commands. First write down the mathematical operation, then identify the numerical algorithm, and only then implement it in Python.


## Suggested 3-hour tutorial structure

| Time | Activity |
|---|---|
| 0–20 min | Mathematical recap |
| 20–45 min | NumPy arrays and vectorization |
| 45–75 min | Guided exercises |
| 75–90 min | Linear algebra |
| 90–100 min | Break |
| 100–140 min | Eigenvalues and SVD |
| 140–170 min | Materials mini-problem |
| 170–180 min | Challenge + reflection |

### Prerequisites

You should know basic algebra, vectors, matrices, and introductory materials science. No previous Python experience beyond Module 1 is assumed.


# 1. Setup

We will use:

- `numpy` for numerical arrays and linear algebra
- `matplotlib` for visualization
- `time` for a simple performance comparison

Run the next cell.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=4, suppress=True)

print("NumPy version:", np.__version__)


# 2. NumPy arrays: mathematical vectors and grids

A mathematical vector can be represented as

\[
\mathbf{x} =
\begin{bmatrix}
x_1 & x_2 & \cdots & x_n
\end{bmatrix}.
\]

NumPy arrays provide a compact numerical representation of such objects.

## Exercise 2.1 — Create a materials vector

Create a NumPy array containing six temperatures:

\[
T = [300, 400, 500, 600, 700, 800]\;K.
\]

Then determine:

1. Number of elements
2. Shape
3. Data type
4. Mean temperature
5. Maximum temperature
6. Minimum temperature


In [ ]:
# YOUR CODE HERE

T = np.array([300, 400, 500, 600, 700, 800])

print("T =", T)
print("Number of elements =", T.size)
print("Shape =", T.shape)
print("Data type =", T.dtype)
print("Mean =", T.mean())
print("Maximum =", T.max())
print("Minimum =", T.min())


### Exercise 2.2 — Create a spatial grid

For numerical modelling, we often discretize space:

\[
x_i = x_0 + i\Delta x.
\]

Create 101 equally spaced points from 0 to 10 mm using `np.linspace`.

Then calculate

\[
f(x)=\sin(x)
\]

and plot it.

**Question:** Why is an array representation of the spatial grid useful for finite-difference calculations later in the course?


In [ ]:
# YOUR CODE HERE

x = np.linspace(0, 10, 101)
f = np.sin(x)

plt.figure(figsize=(8, 4))
plt.plot(x, f)
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.title("Function sampled on a numerical grid")
plt.grid()
plt.show()


# 3. Indexing and slicing

For an array

\[
x=[x_0,x_1,\ldots,x_N],
\]

Python uses zero-based indexing.

Practice extracting:

- first value
- last value
- first five values
- every second value
- interior points excluding the two boundaries

The last operation is particularly important for finite-difference methods because boundary points are often treated separately.


In [ ]:
# Complete the following

print("First value:", x[0])
print("Last value:", x[-1])
print("First five:", x[:5])
print("Every second value:", x[::2])
print("Interior points:", x[1:-1])


# 4. Broadcasting

Suppose the linear thermal expansion model is

\[
L(T)=L_0[1+\alpha(T-T_0)].
\]

If `T` is an array, NumPy can evaluate the equation for every temperature without an explicit loop.

This is called **broadcasting/vectorized computation**.


In [ ]:
L0 = 10.0       # mm
alpha = 23e-6    # 1/K
T0 = 300.0       # K

T = np.linspace(300, 1000, 100)

L = L0 * (1 + alpha * (T - T0))

plt.figure(figsize=(8, 4))
plt.plot(T, L)
plt.xlabel("Temperature (K)")
plt.ylabel("Length (mm)")
plt.title("Thermal expansion")
plt.grid()
plt.show()


## Exercise 4.1 — Vectorized stress calculation

For

\[
\sigma = \frac{F}{A},
\]

calculate stress for forces from 0 to 10,000 N for a constant cross-sectional area of 25 mm².

Plot stress versus force.

**Extension:** Repeat the calculation for three different areas and plot all three curves.


In [ ]:
# YOUR CODE HERE

F = np.linspace(0, 10000, 100)
A = 25.0  # mm^2

sigma = F / A

plt.figure(figsize=(8, 4))
plt.plot(F, sigma)
plt.xlabel("Force (N)")
plt.ylabel("Stress (N/mm²)")
plt.title("Stress versus applied force")
plt.grid()
plt.show()


# 5. Vectorization versus Python loops

Scientific simulations often perform the same operation millions of times.

Let's compare a Python loop with NumPy vectorization.

We calculate

\[
y = 3x^2+2x+1.
\]

The mathematical operation is identical; the computational implementation is different.


In [ ]:
N = 1_000_000
x = np.linspace(0, 10, N)

# Python loop
start = time.perf_counter()

y_loop = np.empty(N)
for i in range(N):
    y_loop[i] = 3*x[i]**2 + 2*x[i] + 1

loop_time = time.perf_counter() - start

# NumPy vectorization
start = time.perf_counter()

y_numpy = 3*x**2 + 2*x + 1

numpy_time = time.perf_counter() - start

print(f"Loop time:       {loop_time:.4f} s")
print(f"NumPy time:      {numpy_time:.4f} s")
print(f"Speedup:         {loop_time/numpy_time:.1f}x")
print("Results identical:", np.allclose(y_loop, y_numpy))


## Reflection

1. Why is vectorization usually faster?
2. In what situations might a loop still be useful?
3. Why is vectorization especially important for finite-difference grids and large materials datasets?


# 6. Matrices

A matrix is a two-dimensional numerical object:

\[
A=
\begin{bmatrix}
a_{11}&a_{12}&a_{13}\\
a_{21}&a_{22}&a_{23}\\
a_{31}&a_{32}&a_{33}
\end{bmatrix}.
\]

Create the matrix below:

\[
A=
\begin{bmatrix}
4&1&2\\
1&5&0\\
2&0&3
\end{bmatrix}.
\]


In [ ]:
A = np.array([
    [4, 1, 2],
    [1, 5, 0],
    [2, 0, 3]
], dtype=float)

print(A)
print("Shape:", A.shape)


# 7. Matrix operations

## 7.1 Transpose

The transpose is

\[
A^T.
\]

## 7.2 Matrix multiplication

For compatible matrices,

\[
C=AB.
\]

Be careful: element-wise multiplication and matrix multiplication are different operations.

- `A * B` → element-wise multiplication
- `A @ B` → matrix multiplication


In [ ]:
A_T = A.T
A2 = A @ A

print("A^T =\n", A_T)
print("\nA @ A =\n", A2)


## Exercise 7.1

Let

\[
B=
\begin{bmatrix}
1&2\\
3&4
\end{bmatrix}.
\]

Calculate:

1. \(B^T\)
2. \(B^TB\)
3. \(BB^T\)
4. \(B\times B\) element-wise

Explain why \(B^TB\) and \(BB^T\) are different matrices.


In [ ]:
B = np.array([
    [1, 2],
    [3, 4]
], dtype=float)

# YOUR CODE HERE
print("B^T =\n", B.T)
print("B^T B =\n", B.T @ B)
print("B B^T =\n", B @ B.T)
print("Element-wise B*B =\n", B * B)


# 8. Determinant, inverse, rank and norm

For a square matrix:

- determinant: \(\det(A)\)
- inverse: \(A^{-1}\), if it exists
- rank: number of linearly independent rows/columns
- vector norm: a measure of vector magnitude

Use NumPy to calculate these quantities.


In [ ]:
det_A = np.linalg.det(A)
inv_A = np.linalg.inv(A)
rank_A = np.linalg.matrix_rank(A)

v = np.array([3, 4])
norm_v = np.linalg.norm(v)

print("det(A) =", det_A)
print("\nA^-1 =\n", inv_A)
print("\nrank(A) =", rank_A)
print("\n||v|| =", norm_v)

print("\nCheck A A^-1 ≈ I:")
print(A @ inv_A)


## Exercise 8.1 — Singular matrix

Consider

\[
S=
\begin{bmatrix}
1&2\\
2&4
\end{bmatrix}.
\]

Calculate its determinant and rank.

Try to calculate its inverse.

**Question:** Why does an inverse not exist? Relate this to linear dependence.


In [ ]:
S = np.array([
    [1, 2],
    [2, 4]
], dtype=float)

print("det(S) =", np.linalg.det(S))
print("rank(S) =", np.linalg.matrix_rank(S))

# Try:
# print(np.linalg.inv(S))


# 9. Orthogonality

Two vectors are orthogonal if

\[
\mathbf{u}\cdot\mathbf{v}=0.
\]

For numerical work, use a tolerance rather than expecting exactly zero.

Consider:

\[
u=(1,2,3),\qquad
v=(2,-1,0).
\]

Check their dot product and determine whether they are orthogonal.


In [ ]:
u = np.array([1, 2, 3], dtype=float)
v = np.array([2, -1, 0], dtype=float)

dot_uv = np.dot(u, v)

print("u · v =", dot_uv)
print("Orthogonal:", np.isclose(dot_uv, 0.0))


## Challenge 9.1

Construct a third vector that is orthogonal to both

\[
u=(1,2,3),\quad v=(2,-1,0).
\]

**Hint:** Try the cross product.

Then verify the two dot products numerically.


In [ ]:
w = np.cross(u, v)

print("w =", w)
print("u · w =", np.dot(u, w))
print("v · w =", np.dot(v, w))


# 10. Solving linear systems

Many scientific problems reduce to

\[
A\mathbf{x}=\mathbf{b}.
\]

Example:

\[
\begin{aligned}
4x+y+2z &= 12\\
x+5y &= 7\\
2x+3z &= 10.
\end{aligned}
\]

The preferred numerical operation is generally to solve the system directly rather than explicitly calculating \(A^{-1}\).


In [ ]:
A_sys = np.array([
    [4, 1, 2],
    [1, 5, 0],
    [2, 0, 3]
], dtype=float)

b_sys = np.array([12, 7, 10], dtype=float)

x_sys = np.linalg.solve(A_sys, b_sys)

print("Solution x =", x_sys)
print("Residual A x - b =", A_sys @ x_sys - b_sys)
print("Residual norm =", np.linalg.norm(A_sys @ x_sys - b_sys))


## Materials interpretation

Systems of equations appear in many computational materials problems, including:

- coupled heat-transfer models
- finite-element/finite-difference discretizations
- fitting constitutive parameters
- force-balance equations
- transport problems
- coupled concentration equations

The important idea is:

\[
\text{continuous physical model}
\rightarrow
\text{discretization}
\rightarrow
A\mathbf{x}=\mathbf{b}.
\]

We will use this idea again in the numerical-methods module.


# 11. Least squares

Experimental data rarely satisfy a model exactly.

Suppose

\[
y=ax+b
\]

and measurements contain noise.

We seek parameters \(a,b\) that minimize

\[
\sum_i(y_i-ax_i-b)^2.
\]

Create a noisy dataset and fit a straight line.


In [ ]:
rng = np.random.default_rng(42)

x_data = np.linspace(0, 10, 20)
y_true = 2.5 * x_data + 4
y_data = y_true + rng.normal(0, 2, size=x_data.size)

# Design matrix
X = np.column_stack([x_data, np.ones_like(x_data)])

# Least-squares solution
beta, residuals, rank, singular_values = np.linalg.lstsq(X, y_data, rcond=None)

slope, intercept = beta

print("Slope =", slope)
print("Intercept =", intercept)
print("Rank =", rank)
print("Singular values =", singular_values)


In [ ]:
y_fit = slope * x_data + intercept

plt.figure(figsize=(8, 4))
plt.scatter(x_data, y_data, label="Synthetic data")
plt.plot(x_data, y_fit, label="Least-squares fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Least-squares fitting")
plt.legend()
plt.grid()
plt.show()


# 12. Eigenvalues and eigenvectors

An eigenvalue problem is

\[
A\mathbf{v}=\lambda\mathbf{v}.
\]

For materials science, eigenvalue problems appear in:

- vibrational modes
- stability analysis
- diffusion tensors
- elasticity
- quantum-mechanical models
- principal component analysis

Calculate the eigenvalues and eigenvectors of \(A\).


In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(A)

print("Eigenvalues:")
print(eigenvalues)

print("\nEigenvectors (columns):")
print(eigenvectors)


## Verify an eigenpair

For each eigenpair, verify that

\[
A\mathbf{v}_i-\lambda_i\mathbf{v}_i\approx0.
\]

This is an important scientific-computing habit:

> Never trust a numerical result blindly. Verify it against the mathematical definition whenever possible.


In [ ]:
for i in range(len(eigenvalues)):
    lam = eigenvalues[i]
    vec = eigenvectors[:, i]
    residual = A @ vec - lam * vec

    print(f"Eigenpair {i+1}")
    print("lambda =", lam)
    print("residual norm =", np.linalg.norm(residual))
    print()


# 13. Eigenvalues of a diffusion tensor

A second-rank property tensor can be written as

\[
D=
\begin{bmatrix}
D_{xx}&D_{xy}&D_{xz}\\
D_{yx}&D_{yy}&D_{yz}\\
D_{zx}&D_{zy}&D_{zz}
\end{bmatrix}.
\]

For a symmetric diffusion tensor, the eigenvalues represent principal diffusivities and eigenvectors give their principal directions.

Consider:

\[
D=
\begin{bmatrix}
2.0&0.4&0.0\\
0.4&1.2&0.0\\
0.0&0.0&0.5
\end{bmatrix}
\times10^{-10}\;m^2/s.
\]

Calculate its principal diffusivities and directions.


In [ ]:
D_tensor = np.array([
    [2.0, 0.4, 0.0],
    [0.4, 1.2, 0.0],
    [0.0, 0.0, 0.5]
])

eig_D, vec_D = np.linalg.eigh(D_tensor)

print("Principal diffusivities (x 1e-10 m^2/s):")
print(eig_D)

print("\nPrincipal directions (columns):")
print(vec_D)


# 14. Singular Value Decomposition (SVD)

SVD decomposes a matrix as

\[
A=U\Sigma V^T.
\]

The singular values in \(\Sigma\) tell us how much information is associated with each orthogonal direction.

SVD is important for:

- low-rank approximation
- noise filtering
- dimensional reduction
- inverse problems
- PCA-related computations
- compressing scientific datasets



In [ ]:
U, s, Vt = np.linalg.svd(A)

print("U =\n", U)
print("\nSingular values =", s)
print("\nV^T =\n", Vt)


## Verify the decomposition

The diagonal matrix \(\Sigma\) must be constructed from the singular values.

Then verify:

\[
A\approx U\Sigma V^T.
\]


In [ ]:
Sigma = np.zeros_like(A)
np.fill_diagonal(Sigma, s)

A_reconstructed = U @ Sigma @ Vt

print("Reconstructed A =\n", A_reconstructed)
print("\nReconstruction error =", np.linalg.norm(A - A_reconstructed))


# 15. Low-rank approximation

Keep only the largest singular values.

For a rank-\(k\) approximation:

\[
A_k=U_k\Sigma_kV_k^T.
\]

## Exercise

Construct rank-1 and rank-2 approximations of a matrix and compare the errors.

Then answer:

1. Which singular values dominate?
2. What information is lost?
3. When could low-rank approximation be useful in computational materials?


In [ ]:
def low_rank_approximation(A, k):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

for k in [1, 2, 3]:
    A_k = low_rank_approximation(A, k)
    error = np.linalg.norm(A - A_k)
    print(f"Rank {k} approximation error = {error:.6f}")


# 16. SVD on a synthetic materials dataset

Suppose rows represent different materials and columns represent correlated descriptors:

- atomic size
- density
- melting point
- elastic modulus
- thermal conductivity

Create a synthetic dataset with correlated features.

We will standardize it and examine its singular values.


In [ ]:
rng = np.random.default_rng(7)

n_materials = 100

atomic_size = rng.normal(1.4, 0.1, n_materials)
density = 2.0 * atomic_size + rng.normal(0, 0.05, n_materials)
melting_point = 1200 * atomic_size + rng.normal(0, 40, n_materials)
modulus = 200 * atomic_size + rng.normal(0, 8, n_materials)
thermal_conductivity = 50 * atomic_size + rng.normal(0, 2, n_materials)

X_materials = np.column_stack([
    atomic_size,
    density,
    melting_point,
    modulus,
    thermal_conductivity
])

# Standardize columns
X_centered = X_materials - X_materials.mean(axis=0)
X_scaled = X_centered / X_centered.std(axis=0)

U_m, s_m, Vt_m = np.linalg.svd(X_scaled, full_matrices=False)

explained_variance_ratio = s_m**2 / np.sum(s_m**2)

print("Singular values:")
print(s_m)

print("\nExplained variance ratio:")
print(explained_variance_ratio)

print("\nCumulative explained variance:")
print(np.cumsum(explained_variance_ratio))


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    range(1, len(explained_variance_ratio) + 1),
    np.cumsum(explained_variance_ratio),
    marker="o"
)
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("Information captured by SVD/PCA components")
plt.ylim(0, 1.05)
plt.grid()
plt.show()


# 17. Optional advanced topic — QR decomposition

QR decomposition writes

\[
A=QR
\]

where:

- \(Q\) contains orthonormal columns
- \(R\) is upper triangular

QR is important in numerical linear algebra and is often preferable to explicitly forming matrix inverses in numerical algorithms.

Use NumPy to calculate and verify a QR decomposition.


In [ ]:
Q, R = np.linalg.qr(A)

print("Q =\n", Q)
print("\nR =\n", R)

print("\nQ^T Q =\n", Q.T @ Q)
print("\nReconstruction error =", np.linalg.norm(A - Q @ R))


# 18. Integrated Materials Problem — Thermal Expansion Tensor

Consider a simplified anisotropic thermal expansion tensor:

\[
\boldsymbol{\alpha}=
\begin{bmatrix}
20&3&0\\
3&10&0\\
0&0&5
\end{bmatrix}
\times10^{-6}\;K^{-1}.
\]

For a temperature change \(\Delta T=100\;K\), the linearized thermal strain for a unit direction \(\mathbf{n}\) is related to the tensor through

\[
\epsilon_n \sim \mathbf{n}^T\boldsymbol{\alpha}\mathbf{n}\Delta T.
\]

### Tasks

1. Construct the tensor.
2. Verify that it is symmetric.
3. Calculate its eigenvalues/eigenvectors.
4. Identify the principal thermal expansion coefficients.
5. Calculate the thermal strain along a chosen unit direction.
6. Compare the result with the principal-direction values.
7. Explain why eigenvectors are useful for anisotropic materials.


In [ ]:
alpha = np.array([
    [20, 3, 0],
    [3, 10, 0],
    [0, 0, 5]
], dtype=float) * 1e-6

delta_T = 100.0

print("Symmetric:", np.allclose(alpha, alpha.T))

alpha_values, alpha_vectors = np.linalg.eigh(alpha)

print("\nPrincipal thermal expansion coefficients (1/K):")
print(alpha_values)

print("\nPrincipal directions:")
print(alpha_vectors)


In [ ]:
# Choose a unit direction
n = np.array([1.0, 1.0, 0.0])
n = n / np.linalg.norm(n)

epsilon_n = n @ alpha @ n * delta_T

print("Unit direction n =", n)
print("Thermal strain along n =", epsilon_n)


# 19. Numerical verification challenge

For the thermal-expansion tensor, verify the eigenvalue equation for every eigenpair:

\[
\alpha\mathbf{v}_i
=
\lambda_i\mathbf{v}_i.
\]

Report the residual norm for each pair.

Then verify that the eigenvectors are mutually orthogonal.


In [ ]:
# YOUR CODE HERE

for i in range(3):
    lam = alpha_values[i]
    vec = alpha_vectors[:, i]
    residual = alpha @ vec - lam * vec
    print(f"Eigenpair {i+1}: residual norm = {np.linalg.norm(residual):.3e}")

print("\nV^T V =")
print(alpha_vectors.T @ alpha_vectors)


# 20. Mini-project — Build a numerical linear-algebra investigation

Choose **one** of the following.

### Option A — Diffusion tensor

Generate an anisotropic diffusion tensor and determine:

- principal diffusivities
- principal directions
- diffusion along arbitrary directions

### Option B — Elastic stiffness matrix

Construct a simplified symmetric stiffness matrix and investigate:

- eigenvalues
- eigenvectors
- positive definiteness
- condition/rank

### Option C — Experimental calibration

Generate noisy stress-strain data and use least squares to determine:

- Young's modulus
- intercept
- residual error

### Option D — SVD of a materials dataset

Use a provided or self-generated dataset to:

- standardize data
- perform SVD
- calculate explained variance
- construct low-rank approximations
- interpret dominant correlations

## Required report structure

1. Physical question
2. Mathematical formulation
3. Numerical method
4. Python implementation
5. Verification
6. Results
7. Visualization
8. Physical interpretation
9. Limitations


# 21. Assessment questions

## Conceptual

1. Why are NumPy arrays preferable to Python lists for numerical computing?
2. What is broadcasting?
3. What is the difference between `A * B` and `A @ B`?
4. Why is `np.linalg.solve(A, b)` generally preferable to explicitly computing \(A^{-1}b\)?
5. What does matrix rank tell us?
6. What does orthogonality mean?
7. What is an eigenvector physically in a tensor problem?
8. What information is contained in singular values?
9. Why does SVD help with dimensional reduction?
10. Why should numerical results be verified?

## Coding

Write functions that:

- calculate vector magnitude
- test orthogonality
- solve a linear system
- calculate an eigenpair residual
- calculate a rank-\(k\) SVD approximation

## Scientific reasoning

Explain how the chain

\[
\text{physical model}
\rightarrow
\text{matrix/tensor}
\rightarrow
\text{linear algebra}
\rightarrow
\text{numerical solution}
\]

appears in a computational materials problem of your choice.


# 22. Take-home assignment

## Assignment: Numerical Linear Algebra for an Anisotropic Materials Property

You are given a symmetric 3×3 materials-property tensor.

### Part A — NumPy

- Construct the tensor.
- Inspect shape and symmetry.
- Calculate determinant, rank and norm.

### Part B — Linear algebra

- Calculate eigenvalues/eigenvectors.
- Verify every eigenpair.
- Check eigenvector orthogonality.

### Part C — Directional property

For at least five unit directions, calculate

\[
p(\mathbf{n})=\mathbf{n}^TA\mathbf{n}.
\]

Visualize the values.

### Part D — SVD

- Calculate the SVD.
- Reconstruct the tensor.
- Calculate rank-1 and rank-2 approximations.
- Compare errors.

### Part E — Interpretation

Write 500–800 words explaining:

- what the eigenvalues mean
- what the eigenvectors mean
- what SVD reveals
- which numerical quantities are physically meaningful
- limitations of the simplified model

### Grading rubric

| Component | Marks |
|---|---:|
| Correct Python implementation | 20 |
| Mathematical formulation | 20 |
| Verification/testing | 15 |
| Numerical analysis | 15 |
| Visualization | 10 |
| Physical interpretation | 15 |
| Code quality/reproducibility | 5 |
| **Total** | **100** |


# 23. Module 2 — What you should now understand

You should be able to move between four representations:

### Physical problem

> An anisotropic material has direction-dependent transport.

↓

### Mathematical representation

\[
\mathbf{J}=-D\nabla c
\]

where \(D\) is a tensor.

↓

### Numerical representation

\[
D=
\begin{bmatrix}
D_{xx}&D_{xy}&D_{xz}\\
D_{yx}&D_{yy}&D_{yz}\\
D_{zx}&D_{zy}&D_{zz}
\end{bmatrix}.
\]

↓

### Python representation

```python
D = np.array([
    [Dxx, Dxy, Dxz],
    [Dyx, Dyy, Dyz],
    [Dzx, Dzy, Dzz]
])
```

That transition — **physics → mathematics → numerical algorithm → Python** — is the core skill of this module.

---

## Next module connection

Module 3 will build directly on these skills:

**NumPy arrays → Pandas DataFrames → scientific visualization → statistical analysis of real materials datasets.**

Later, PCA will return to the linear algebra introduced here, and finite-difference methods will use NumPy arrays to discretize differential equations.
